[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lugerlakes/signal_analysis_course/blob/main/Lab6_Integrador_Informe_ELINT_ESM.ipynb)

<div style="border-left:6px solid #1F4E79;padding:10px 16px;background:#F3F7FB">

<span style="font-size:12px;letter-spacing:1px;color:#1F4E79">CURSO DE GUERRA ELECTRÓNICA · ASIGNATURA: ANÁLISIS DE SEÑALES DE RADAR</span>

# Laboratorio 6 · Integrador: Informe Preliminar de Interceptación ELINT/ESM
### De los PDW crudos de una misión al informe para el mando

<span style="font-size:12px;color:#7a7a7a">MATERIAL DE INSTRUCCIÓN · DATOS SINTÉTICOS · Emisores y bibliotecas ficticios</span>
</div>

| | |
|---|---|
| ⏱️ **Duración estimada** | 4 a 6 horas (trabajo en grupos de 4) |
| 📥 **Datos de entrada** | `D5_MISION_GRUPO_<n>.xlsx` (una misión distinta por grupo) · `BIBLIOTECA_EMISORES_ficticia.xlsx` |
| 📤 **Producto al final** | **IPI** completo (en este cuaderno) + Anexo técnico (Excel) + figuras |

**Objetivos de aprendizaje** — al terminar este laboratorio serás capaz de:

1. Ejecutar de forma autónoma el **ciclo completo de producción**: preprocesamiento → extracción de características → identificación → reporte.
2. Separar emisores con **GMM**, depurar ruido y **asociar modos** de un mismo emisor.
3. Identificar emisores contra la biblioteca con **nivel de confianza** y documentar los **desconocidos**.
4. Reconstruir la **cronología de eventos** en hora UTC y evaluar el **nivel de amenaza**.
5. Redactar un **Informe Preliminar de Interceptación (IPI)** claro, trazable y con incertidumbres explícitas.

**Cómo leer este cuaderno**

| Ícono | Significado |
|---|---|
| 📘 | Concepto o teoría breve |
| ▶️ | Celda de código para **ejecutar** (clic en la celda y `Shift + Enter`) |
| ✏️ | **Tu turno**: completa el código o responde |
| 💡 | Consejo práctico |
| ⚠️ | Error frecuente o precaución |
| 🎯 | Conexión con la Guerra Electrónica (ESM / ELINT) |

> Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla de forma extraña: menú **Entorno de ejecución → Reiniciar y ejecutar todo**.

---
## 0. Briefing de la misión

| Grupo | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| Archivo | `D5_MISION_GRUPO_1.xlsx` | `D5_MISION_GRUPO_2.xlsx` | `D5_MISION_GRUPO_3.xlsx` | `D5_MISION_GRUPO_4.xlsx` |

Cada archivo contiene una **salida de colección** simulada de 60 s de una plataforma ISR con receptor ESM. Hoja `METADATOS`: datos de la misión. Hoja `PDW`: pulsos interceptados.

**Tu tarea:** producir un **Informe Preliminar de Interceptación (IPI)**. "Preliminar" significa: rápido, basado en el análisis inicial, con **incertidumbres declaradas**, sujeto a confirmación por el análisis ELINT detallado.

📘 **ESM vs ELINT en el producto final**
| | Reporte ESM (táctico) | Informe ELINT (técnico) |
|---|---|---|
| Plazo | Segundos / minutos | Horas / días |
| Contenido | Qué emisor, dónde, qué amenaza, qué hace ahora | Parámetros detallados, modos, novedades, actualización de biblioteca |
| Destinatario | Tripulación, mando táctico | Centro de análisis, mando, planificación EW |

El IPI de este laboratorio combina ambos: **resumen para el mando** (BLUF) + **anexo técnico**.

▶️ Elige tu grupo y ejecuta la caja de herramientas:

In [ ]:
GRUPO = 1          # ✏️ 1, 2, 3 ó 4
ARCHIVO = f"D5_MISION_GRUPO_{GRUPO}.xlsx"
print("Misión asignada:", ARCHIVO)

In [ ]:
# @title 🧰 Caja de herramientas completa (Labs 1–6) — ejecutar
# ------------------------------------------------------------------
# A. CARGA Y LIMPIEZA DE PDW (lo que aprendiste a mano en el Lab 1)
# ------------------------------------------------------------------
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 4)

CARPETAS_DATOS = [".", "datos", "/content", "/content/datos", "/content/sample_data",
                  "/content/drive/MyDrive", "/content/drive/MyDrive/datos",
                  "/content/drive/MyDrive/Analisis_de_Señales"]


def buscar_archivo(nombre):
    """Busca 'nombre' en las carpetas habituales (local y Google Colab)."""
    for carpeta in CARPETAS_DATOS:
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    raise FileNotFoundError(
        f"No encontré '{nombre}'. Súbelo con el ícono de carpeta 📁 (panel izquierdo de Colab) "
        f"o copia su ruta exacta en la variable del archivo.")


def estandarizar_columnas(df):
    """Renombra las columnas de la exportación ESM a nombres estándar con unidades."""
    nuevos = {}
    for col in df.columns:
        c = str(col).upper()
        if c.startswith("UNNAMED") or c.strip() == "":
            continue
        if "TOA" in c:
            nuevos[col] = "TOA_us"
        elif "FREQ" in c or "FREC" in c:
            nuevos[col] = "FREQ_MHz"
        elif c.startswith("PW") or "WIDTH" in c:
            nuevos[col] = "PW_us"
        elif "AMP" in c:
            nuevos[col] = "AMP_dBm"
        elif "AOA" in c or "DOA" in c or "BEARING" in c:
            nuevos[col] = "AOA_deg"
        elif "PRI" in c:
            nuevos[col] = "PRI_us"
    columnas_vacias = [c for c in df.columns if c not in nuevos]
    return df.drop(columns=columnas_vacias).rename(columns=nuevos)


def limpiar_pdw(df, verbose=True):
    """Limpieza estándar de PDW: centinelas, vacíos, duplicados y orden por TOA."""
    n0 = len(df)
    df = df.copy()
    claves = [c for c in ["TOA_us", "FREQ_MHz", "PW_us"] if c in df.columns]
    df = df.dropna(subset=claves)
    n1 = len(df)
    malos = (df["FREQ_MHz"] <= 0) | (df["PW_us"] <= 0)      # registros "fantasma" del receptor
    df = df[~malos]
    n2 = len(df)
    df = df.drop_duplicates()
    n3 = len(df)
    df = df.sort_values("TOA_us").reset_index(drop=True)
    df["TOA_s"] = (df["TOA_us"] - df["TOA_us"].iloc[0]) / 1e6
    df["DTOA_us"] = df["TOA_us"].diff()
    if verbose:
        print(f"Filas originales ............ {n0:>7,}")
        print(f"  − con celdas vacías ....... {n0 - n1:>7,}")
        print(f"  − con ceros / centinela ... {n1 - n2:>7,}")
        print(f"  − duplicadas .............. {n2 - n3:>7,}")
        print(f"Filas válidas ............... {n3:>7,}")
    return df


# Carpeta web con los datos del curso (repositorio público). Déjala vacía ("") para trabajar sólo con archivos locales.
URL_DATOS = "https://raw.githubusercontent.com/lugerlakes/signal_analysis_course/main/datos/"


def asegurar_archivo(nombre):
    """Devuelve la ruta del archivo; si no está, lo descarga (URL_DATOS) o pide subirlo (Colab)."""
    try:
        return buscar_archivo(nombre)
    except FileNotFoundError:
        pass
    if URL_DATOS:
        import urllib.request, urllib.parse
        os.makedirs("datos", exist_ok=True)
        destino = os.path.join("datos", nombre)
        try:
            urllib.request.urlretrieve(URL_DATOS.rstrip("/") + "/" + urllib.parse.quote(nombre), destino)
            print(f"⬇️ '{nombre}' descargado desde el repositorio del curso")
            return destino
        except Exception as e:
            print(f"⚠️ No se pudo descargar '{nombre}' ({e}). Intentando otras opciones…")
            if os.path.exists(destino):
                os.remove(destino)
    try:
        from google.colab import files
        print(f"📁 No encuentro '{nombre}'. Selecciónalo desde tu computador:")
        subidos = files.upload()
        if nombre in subidos:
            return nombre
    except ImportError:
        pass
    return buscar_archivo(nombre)   # mensaje de error claro


def cargar_pdw(nombre_archivo, hoja=0, verbose=True):
    """Carga un archivo ESM (Excel o CSV), estandariza columnas y limpia."""
    ruta = asegurar_archivo(nombre_archivo)
    if ruta.lower().endswith(".csv"):
        crudo = pd.read_csv(ruta)
    else:
        crudo = pd.read_excel(ruta, sheet_name=hoja)
    df = limpiar_pdw(estandarizar_columnas(crudo), verbose=verbose)
    if verbose:
        dur = df["TOA_s"].iloc[-1]
        print(f"Duración de la captura ...... {dur:,.2f} s")
    return df
# ------------------------------------------------------------------
# B. ESTADÍSTICA DESCRIPTIVA (Lab 2)
# ------------------------------------------------------------------
def moda_continua(serie, bins="auto"):
    """Moda de una variable continua = centro de la barra más alta del histograma."""
    x = pd.Series(serie).dropna().values
    cuentas, bordes = np.histogram(x, bins=bins)
    i = np.argmax(cuentas)
    return 0.5 * (bordes[i] + bordes[i + 1])


def ic95_media(serie, confianza=0.95):
    """Intervalo de confianza para la MEDIA usando la distribución t de Student."""
    x = pd.Series(serie).dropna().values
    n = len(x)
    media = x.mean()
    ee = x.std(ddof=1) / np.sqrt(n)              # error estándar de la media
    t = stats.t.ppf(0.5 + confianza / 2, df=n - 1)
    return media - t * ee, media + t * ee


def ic95_mediana_bootstrap(serie, n_boot=2000, confianza=0.95, semilla=0):
    """Intervalo de confianza para la MEDIANA por remuestreo (bootstrap)."""
    x = pd.Series(serie).dropna().values
    rng = np.random.default_rng(semilla)
    medianas = np.median(rng.choice(x, size=(n_boot, len(x)), replace=True), axis=1)
    a = (1 - confianza) / 2
    return np.quantile(medianas, a), np.quantile(medianas, 1 - a)


def resumen_estadistico(serie, nombre=None):
    """Tabla con los estadígrafos más usados en un informe de análisis."""
    x = pd.Series(serie).dropna()
    lo, hi = ic95_media(x)
    return pd.Series({
        "n": int(len(x)),
        "media": x.mean(),
        "IC95_media_inf": lo,
        "IC95_media_sup": hi,
        "mediana": x.median(),
        "moda (histograma)": moda_continua(x),
        "desv_estandar": x.std(ddof=1),
        "CV_%": 100 * x.std(ddof=1) / abs(x.mean()) if x.mean() != 0 else np.nan,
        "minimo": x.min(),
        "P5": x.quantile(0.05),
        "Q1 (P25)": x.quantile(0.25),
        "Q3 (P75)": x.quantile(0.75),
        "P95": x.quantile(0.95),
        "maximo": x.max(),
        "rango": x.max() - x.min(),
        "IQR": x.quantile(0.75) - x.quantile(0.25),
        "asimetria": stats.skew(x),
    }, name=nombre or getattr(serie, "name", "variable"))


def graficar_distribucion(serie, titulo="", unidad="", bins=60):
    """Histograma + KDE, boxplot y violín de una misma variable (3 vistas)."""
    x = pd.Series(serie).dropna()
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw={"width_ratios": [2.2, 1, 1]})
    sns.histplot(x, bins=bins, kde=True, ax=ax[0], color="#4C72B0")
    ax[0].axvline(x.mean(), color="#C44E52", ls="--", lw=1.5, label=f"media = {x.mean():.4g}")
    ax[0].axvline(x.median(), color="#55A868", ls="-", lw=1.5, label=f"mediana = {x.median():.4g}")
    ax[0].legend(fontsize=9)
    ax[0].set_title(f"Histograma + KDE · {titulo}")
    ax[0].set_xlabel(unidad)
    sns.boxplot(y=x, ax=ax[1], color="#8172B2", width=0.4)
    ax[1].set_title("Boxplot"); ax[1].set_ylabel(unidad)
    sns.violinplot(y=x, ax=ax[2], color="#64B5CD", inner="quartile", cut=0)
    ax[2].set_title("Violín"); ax[2].set_ylabel(unidad)
    plt.tight_layout()
    plt.show()
# ------------------------------------------------------------------
# C. CARACTERIZACIÓN DE UN EMISOR (Lab 3)
# ------------------------------------------------------------------
from scipy.signal import find_peaks
from collections import Counter

C_LUZ = 299_792_458.0  # m/s


def banda_ieee(f_mhz):
    f = f_mhz / 1000.0
    for lim, b in [(0.3, "VHF"), (1, "UHF"), (2, "L"), (4, "S"), (8, "C"), (12, "X"),
                   (18, "Ku"), (27, "K"), (40, "Ka"), (75, "V"), (110, "W")]:
        if f < lim:
            return b
    return "mm"


def banda_otan(f_mhz):
    f = f_mhz / 1000.0
    for lim, b in [(0.25, "A"), (0.5, "B"), (1, "C"), (2, "D"), (3, "E"), (4, "F"), (6, "G"),
                   (8, "H"), (10, "I"), (20, "J"), (40, "K"), (60, "L"), (100, "M")]:
        if f < lim:
            return b
    return ">M"


def analizar_pri(toa_us, verbose=True):
    """Analiza la secuencia de llegadas de UN emisor y sugiere el tipo de PRI.

    Pasos: (1) DTOA = diferencias de TOA; (2) PRI base = valor más bajo frecuente;
    (3) se descartan huecos de iluminación y pulsos perdidos (DTOA > 1.8·PRI base);
    (4) se buscan niveles discretos y se mide la correlación pulso a pulso."""
    toa = np.sort(np.asarray(toa_us, float))
    d = np.diff(toa)
    d = d[d > 0]
    r = {"n_dtoa": len(d)}
    if len(d) < 10:
        r["tipo"] = "Insuficientes pulsos"
        return r
    # (2) PRI base sobre escala logarítmica (bins de 0.5 %)
    ld = np.log(d)
    lb = np.arange(ld.min(), np.quantile(ld, 0.98) + 0.01, 0.005)
    if len(lb) < 3:
        lb = np.linspace(ld.min(), ld.max() + 1e-6, 5)
    cnt, _ = np.histogram(ld, bins=lb)
    acum = np.convolve(cnt, np.ones(7), mode="same")
    i0 = int(np.argmax(acum >= 0.03 * len(d)))
    base = float(np.exp(lb[i0]))
    principal = (d >= 0.8 * base) & (d <= 1.8 * base)
    m = d[principal]
    r.update(pri_base=base, fraccion_principal=principal.mean(),
             fraccion_perdidos=(d > 1.8 * base).mean())
    med = np.median(m)
    r.update(pri_mediana=med, pri_media=m.mean(), pri_min=np.quantile(m, 0.01),
             pri_max=np.quantile(m, 0.99), pri_std=m.std(ddof=1),
             pri_cv_pct=100 * m.std(ddof=1) / m.mean())
    # (4a) niveles discretos
    ancho = max(0.003 * med, 0.05)
    bordes = np.arange(m.min() - ancho, m.max() + 2 * ancho, ancho)
    h, _ = np.histogram(m, bins=bordes)
    centros = 0.5 * (bordes[1:] + bordes[:-1])
    picos, _ = find_peaks(np.r_[0, h, 0], height=0.08 * h.max(), distance=3)
    picos = picos - 1
    niveles = []
    for p in picos:
        sel = np.abs(m - centros[p]) <= 1.5 * ancho
        if sel.sum():
            niveles.append(float(np.median(m[sel])))
    niveles = sorted(niveles)
    conc = np.mean(np.min(np.abs(m[:, None] - np.array(niveles)[None, :]), axis=1) <= 2 * ancho) if niveles else 0
    discreto = (1 <= len(niveles) <= 8) and conc >= 0.85
    # (4b) correlación entre PRI consecutivos (sólo pares sin pulsos perdidos)
    ok = principal[:-1] & principal[1:]
    a, b = d[:-1][ok], d[1:][ok]
    r1 = float(np.corrcoef(a, b)[0, 1]) if len(a) > 5 and a.std() > 0 and b.std() > 0 else 0.0
    r.update(niveles=niveles, concentracion=conc, autocorr_lag1=r1)
    # (5) clasificación
    if r["fraccion_principal"] < 0.4:
        r["tipo"] = "Sin estructura (posible ruido o mezcla)"
        r["detalle"] = (f"sólo {100 * r['fraccion_principal']:.0f} % de los DTOA forman un patrón: "
                        "no hay un tren de pulsos coherente")
        if verbose:
            print(f"Tipo de PRI sugerido : {r['tipo']}\nDetalle              : {r['detalle']}")
        return r
    if discreto and len(niveles) == 1:
        tipo = "Estable"
        detalle = f"PRI ≈ {niveles[0]:.2f} µs (PRF ≈ {1e6 / niveles[0]:.1f} Hz), CV = {r['pri_cv_pct']:.3f} %"
    elif discreto:
        niv = np.array(niveles)
        idx = np.argmin(np.abs(m[:, None] - niv[None, :]), axis=1)
        corridas = np.diff(np.flatnonzero(np.r_[True, idx[1:] != idx[:-1], True]))
        largo_corrida = corridas.mean()
        if largo_corrida >= 4:
            tipo = "Dwell & Switch"
            detalle = (f"{len(niv)} PRI: " + ", ".join(f"{v:.1f}" for v in niv) +
                       f" µs · ~{largo_corrida:.0f} pulsos por permanencia (dwell)")
            r["pulsos_por_dwell"] = largo_corrida
        else:
            tipo = f"Stagger {len(niv)} posiciones"
            k = len(niv)
            ventanas = [tuple(idx[i:i + k]) for i in range(len(idx) - k)]
            ventanas = [w for w in ventanas if len(set(w)) == k]
            seq = Counter(ventanas).most_common(1)[0][0] if ventanas else tuple(range(k))
            orden = [niv[j] for j in seq]
            r["secuencia"] = orden
            r["periodo_trama"] = float(niv.sum())
            detalle = ("secuencia " + " → ".join(f"{v:.1f}" for v in orden) +
                       f" µs · periodo de trama {niv.sum():.1f} µs · PRI media {niv.mean():.1f} µs")
    else:
        if r1 > 0.6:
            tipo = "Deslizante (sliding)"
            caidas = np.flatnonzero(np.diff(m) < -0.5 * (r["pri_max"] - r["pri_min"]))
            pulsos_rampa = float(np.median(np.diff(caidas))) if len(caidas) > 2 else np.nan
            r["pulsos_por_rampa"] = pulsos_rampa
            detalle = (f"rampa de {r['pri_min']:.0f} a {r['pri_max']:.0f} µs, ~{pulsos_rampa:.0f} pulsos por rampa")
        else:
            tipo = "Jitter (aleatorio)"
            pct = 100 * (r["pri_max"] - r["pri_min"]) / (2 * med)
            r["jitter_pct"] = pct
            detalle = f"PRI central ≈ {med:.1f} µs, variación ≈ ±{pct:.1f} %"
    r["tipo"] = tipo
    r["detalle"] = detalle
    if verbose:
        print(f"Tipo de PRI sugerido : {tipo}")
        print(f"Detalle              : {detalle}")
        print(f"Pulsos perdidos      : {100 * r['fraccion_perdidos']:.1f} % de los DTOA (múltiplos de la PRI o huecos de scan)")
    return r


def graficar_pri(toa_us, titulo="", max_pulsos=600):
    """Dos vistas clásicas: secuencia DTOA vs nº de pulso, e histograma de DTOA."""
    d = np.diff(np.sort(np.asarray(toa_us, float)))
    lim = np.quantile(d, 0.97) * 1.3
    fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
    n = min(len(d), max_pulsos)
    ax[0].plot(np.arange(n), d[:n], ".", ms=4)
    ax[0].set_ylim(0, lim)
    ax[0].set_xlabel("Nº de pulso"); ax[0].set_ylabel("DTOA (µs)")
    ax[0].set_title(f"Secuencia de PRI · {titulo}")
    ax[1].hist(d[d < lim], bins=200, color="#4C72B0")
    ax[1].set_xlabel("DTOA (µs)"); ax[1].set_ylabel("Nº de pulsos")
    ax[1].set_title("Histograma de DTOA")
    plt.tight_layout(); plt.show()


def detectar_iluminaciones(df, gap_us=100_000, min_pulsos=5):
    """Agrupa pulsos en 'iluminaciones' (ráfagas) separadas por huecos > gap_us.
    Se ignoran grupos con menos de 'min_pulsos' (pulsos sueltos de lóbulos laterales)."""
    toa = df["TOA_us"].values
    tsec = df["TOA_s"].values if "TOA_s" in df else toa / 1e6     # segundos desde el inicio de la captura
    amp = df["AMP_dBm"].values
    corte = np.r_[0, np.flatnonzero(np.diff(toa) > gap_us) + 1, len(toa)]
    filas = []
    for i, j in zip(corte[:-1], corte[1:]):
        if j - i < min_pulsos:
            continue
        t, a = toa[i:j], amp[i:j]
        sel = a >= a.max() - 3.0                       # pulsos dentro de 3 dB del máximo
        w = 10 ** (a[sel] / 10)
        # ancho del lóbulo principal: ajuste de una parábola (en dB) a los pulsos
        # que están a menos de 10 dB del máximo  →  ancho a -3 dB = 2·sqrt(3/|c2|)
        lob = a >= a.max() - 10.0
        tot = np.nan
        if lob.sum() >= 5:
            tc = (t[lob] - t[lob].mean()) / 1e3                      # ms
            c2 = np.polyfit(tc, a[lob], 2)[0]
            if c2 < 0:
                tot = 2 * np.sqrt(3.0 / -c2)
        ts = tsec[i:j]
        filas.append({
            "t_inicio_s": ts[0], "t_fin_s": ts[-1], "n_pulsos": j - i,
            "duracion_ms": (t[-1] - t[0]) / 1e3,
            "t_centro_s": np.sum(ts[sel] * w) / np.sum(w),
            "amp_max_dBm": a.max(),
            "tot_3dB_ms": tot,
        })
    return pd.DataFrame(filas)


def espectro_amplitud(df, fmax=200.0):
    """Espectro (FFT) de la amplitud: revela modulaciones periódicas (scan cónico)."""
    t = df["TOA_us"].values / 1e6
    a = df["AMP_dBm"].values
    fs = min(1.0 / np.median(np.diff(t)), 4 * fmax)
    tu = np.arange(t[0], t[-1], 1 / fs)
    au = np.interp(tu, t, a)
    au = (au - au.mean()) * np.hanning(len(au))
    esp = np.abs(np.fft.rfft(au)) / len(au)
    f = np.fft.rfftfreq(len(au), 1 / fs)
    sel = (f >= 2) & (f <= fmax)
    return f[sel], esp[sel]


def analizar_scan(df, gap_us=100_000, verbose=True):
    """Clasifica el tipo de exploración de antena a partir de AMP vs TOA."""
    il = detectar_iluminaciones(df, gap_us)
    r = {"n_iluminaciones": len(il)}
    durac_total = (df["TOA_us"].iloc[-1] - df["TOA_us"].iloc[0]) / 1e6
    if len(il) == 0:
        r.update(tipo="Indeterminado", periodo_s=np.nan, detalle="pulsos dispersos, sin iluminaciones definidas")
        if verbose:
            print(f"Tipo de scan sugerido : {r['tipo']}\nDetalle               : {r['detalle']}")
        return r
    if len(il) <= 2 and il["duracion_ms"].max() / 1e3 > 0.6 * durac_total:
        f, e = espectro_amplitud(df)
        k = int(np.argmax(e))
        prominencia = e[k] / (np.median(e) + 1e-12)
        prof = df["AMP_dBm"].quantile(0.95) - df["AMP_dBm"].quantile(0.05)
        if prominencia > 8 and prof > 1.5:
            r.update(tipo="Cónico", frecuencia_scan_Hz=float(f[k]), periodo_s=1 / float(f[k]),
                     profundidad_dB=float(prof),
                     detalle=f"modulación de amplitud a {f[k]:.1f} Hz (periodo {1000 / f[k]:.1f} ms), profundidad ≈ {prof:.1f} dB")
        else:
            r.update(tipo="Fijo (iluminación continua)", periodo_s=0.0,
                     detalle=f"amplitud estable (variación {prof:.1f} dB): posible SEGUIMIENTO / lock-on")
    elif len(il) == 2:
        dt2 = float(il["t_centro_s"].iloc[1] - il["t_centro_s"].iloc[0])
        r.update(tipo="Indeterminado (2 iluminaciones)", periodo_s=dt2,
                 detalle=f"sólo 2 iluminaciones separadas {dt2:.2f} s: periodo probable, BAJA confianza (captura más larga)")
    elif len(il) < 3:
        r.update(tipo="Indeterminado", periodo_s=np.nan, detalle="una sola iluminación: no se puede estimar el periodo")
    else:
        dt = np.diff(il["t_centro_s"].values)
        cv = dt.std() / dt.mean()
        if cv < 0.08:
            T = float(np.median(dt))
            tot = float(il["tot_3dB_ms"].median())
            haz = 360.0 * tot / 1000 / T
            r.update(tipo="Circular", periodo_s=T, rpm=60 / T, tot_3dB_ms=tot, haz_estimado_deg=haz,
                     detalle=f"periodo {T:.2f} s ({60 / T:.1f} rpm) · tiempo en blanco (3 dB) {tot:.1f} ms · haz ≈ {haz:.1f}°")
        else:
            pares = dt[:-1] + dt[1:]
            alterna = np.mean(np.sign(np.diff(dt)[:-1]) != np.sign(np.diff(dt)[1:])) > 0.7
            if alterna and pares.std() / pares.mean() < 0.08:
                T = float(np.median(pares))
                r.update(tipo="Sectorial (bidireccional)", periodo_s=T,
                         detalle=f"iluminaciones en pares alternos ({np.median(dt[::2]):.2f} s / {np.median(dt[1::2]):.2f} s) · periodo de barrido {T:.2f} s")
            else:
                r.update(tipo="Irregular", periodo_s=float(np.median(dt)),
                         detalle="intervalos irregulares: posible cambio de modo, exploración electrónica o varios emisores mezclados")
        largas = il["duracion_ms"] > 5 * il["duracion_ms"].median()
        if largas.any():
            r["detalle"] += f" · ⚠ {int(largas.sum())} iluminación(es) prolongada(s): posible paso a SEGUIMIENTO"
    if verbose:
        print(f"Tipo de scan sugerido : {r['tipo']}")
        print(f"Detalle               : {r['detalle']}")
    return r


def media_circular(angulos_deg):
    """Media de ángulos respetando el salto 359° → 0°."""
    a = np.deg2rad(np.asarray(angulos_deg, float))
    return float(np.rad2deg(np.arctan2(np.sin(a).mean(), np.cos(a).mean())) % 360)


def ficha_emisor(df, nombre="EMISOR", gap_us=100_000, verbose=True):
    """Ficha paramétrica completa de un emisor (RF, PW, PRI, scan, AOA)."""
    f = df["FREQ_MHz"]
    rango_rf = f.quantile(0.99) - f.quantile(0.01)
    agil = rango_rf > 15
    pri = analizar_pri(df["TOA_us"], verbose=False)
    scan = analizar_scan(df, gap_us=gap_us, verbose=False)
    ficha = {
        "EMISOR": nombre,
        "N_PULSOS": len(df),
        "T_INICIO_s": df["TOA_s"].min() if "TOA_s" in df else np.nan,
        "T_FIN_s": df["TOA_s"].max() if "TOA_s" in df else np.nan,
        "RF_MEDIANA_MHz": round(f.median(), 1),
        "RF_MIN_MHz": round(f.quantile(0.01), 1),
        "RF_MAX_MHz": round(f.quantile(0.99), 1),
        "RF_TIPO": "Ágil" if agil else "Fija",
        "BANDA_IEEE": banda_ieee(f.median()),
        "BANDA_OTAN": banda_otan(f.median()),
        "PW_MEDIANA_us": round(df["PW_us"].median(), 2),
        "PW_P5_us": round(df["PW_us"].quantile(0.05), 2),
        "PW_P95_us": round(df["PW_us"].quantile(0.95), 2),
        "PRI_TIPO": pri.get("tipo"),
        "PRI_DETALLE": pri.get("detalle"),
        "PRI_MIN_us": round(pri.get("pri_min", np.nan), 1),
        "PRI_MAX_us": round(pri.get("pri_max", np.nan), 1),
        "PRI_MEDIA_us": round(pri.get("pri_media", np.nan), 1),
        "PRF_MEDIA_Hz": round(1e6 / pri.get("pri_media", np.nan), 1),
        "SCAN_TIPO": scan.get("tipo"),
        "SCAN_PERIODO_s": round(scan.get("periodo_s", np.nan), 3),
        "SCAN_DETALLE": scan.get("detalle"),
        "AOA_MEDIA_deg": round(media_circular(df["AOA_deg"]), 1) if "AOA_deg" in df else np.nan,
        "AMP_MAX_dBm": round(df["AMP_dBm"].max(), 1),
    }
    if verbose:
        for k, v in ficha.items():
            print(f"{k:<16}: {v}")
    return ficha
# ------------------------------------------------------------------
# D. DEINTERLEAVING (Lab 4)
# ------------------------------------------------------------------
def asignar_por_ventanas(df, ventanas, etiqueta="GRUPO"):
    """Asigna cada pulso al primer grupo cuyas ventanas (mín, máx) contengan sus parámetros."""
    df = df.copy()
    df[etiqueta] = "SIN_ASIGNAR"
    for grupo, limites in ventanas.items():
        sel = df[etiqueta] == "SIN_ASIGNAR"
        for col, (lo, hi) in limites.items():
            sel &= df[col].between(lo, hi)
        df.loc[sel, etiqueta] = grupo
    return df


def histograma_diferencias(toa_us, orden_max=4, max_us=3000, bins=600, titulo="", log=False):
    """Histograma de diferencias de TOA de orden 1..k (idea base de CDIF/SDIF)."""
    toa = np.sort(np.asarray(toa_us, float))
    fig, ax = plt.subplots(orden_max, 1, figsize=(13, 2.0 * orden_max), sharex=True)
    bordes = np.linspace(0, max_us, bins + 1)
    centros = 0.5 * (bordes[1:] + bordes[:-1])
    for k in range(1, orden_max + 1):
        d = toa[k:] - toa[:-k]
        h, _ = np.histogram(d, bins=bordes)
        ax[k - 1].fill_between(centros, h + (1 if log else 0), step="mid", color="#4C72B0")
        if log:
            ax[k - 1].set_yscale("log")
        ax[k - 1].set_ylabel(f"orden {k}")
    ax[0].set_title(f"Histogramas de diferencias de TOA · {titulo}")
    ax[-1].set_xlabel("Diferencia de TOA (µs)")
    plt.tight_layout(); plt.show()
# ------------------------------------------------------------------
# E. COTEJO CON BIBLIOTECA DE EMISORES (Lab 5)
# ------------------------------------------------------------------
PESOS = {"RF": 0.35, "PRI": 0.30, "PW": 0.20, "SCAN": 0.15}


def _dentro(valor, lo, hi, tol_rel=0.03):
    margen = tol_rel * max(abs(lo), abs(hi), 1e-9)
    return (lo - margen) <= valor <= (hi + margen)


def comparar_con_biblioteca(ficha, biblioteca, top=3):
    """Compara una ficha de emisor contra cada entrada de la biblioteca.

    Puntaje (0 a 1) = suma de pesos de los parámetros que calzan:
    RF 0.35 · PRI 0.30 · PW 0.20 · SCAN 0.15
    La RF es criterio EXCLUYENTE: si la RF no calza, el puntaje es 0."""
    filas = []
    for _, b in biblioteca.iterrows():
        rf_ok = _dentro(ficha["RF_MEDIANA_MHz"], b["RF_MIN_MHz"], b["RF_MAX_MHz"], 0.005)
        pri_ok = _dentro(ficha["PRI_MEDIA_us"], b["PRI_MIN_us"], b["PRI_MAX_us"])
        pw_ok = _dentro(ficha["PW_MEDIANA_us"], b["PW_MIN_us"], b["PW_MAX_us"], 0.10)
        s_lib = str(b["SCAN_TIPO"]).lower()
        s_obs = str(ficha["SCAN_TIPO"]).split(" ")[0].lower()
        scan_ok = s_obs[:5] in s_lib
        puntaje = (PESOS["RF"] * rf_ok + PESOS["PRI"] * pri_ok + PESOS["PW"] * pw_ok + PESOS["SCAN"] * scan_ok)
        if not rf_ok:
            puntaje = 0.0
        filas.append({"ID_LIB": b["ID_LIB"], "DESIGNACION": b["DESIGNACION"], "FUNCION": b["FUNCION"],
                      "RF": "✔" if rf_ok else "✘", "PRI": "✔" if pri_ok else "✘",
                      "PW": "✔" if pw_ok else "✘", "SCAN": "✔" if scan_ok else "✘",
                      "PUNTAJE": round(puntaje, 2), "AMENAZA": b["AMENAZA"]})
    res = pd.DataFrame(filas).sort_values("PUNTAJE", ascending=False).head(top).reset_index(drop=True)
    return res


def elegir_k_por_bic(bics, ks, tolerancia=0.02):
    """Regla del 'codo': el K más pequeño cuyo BIC está a menos del 2 % del
    recorrido total (máx − mín) respecto del mejor BIC."""
    bics = np.asarray(bics, float)
    umbral = bics.min() + tolerancia * (bics.max() - bics.min())
    return int(np.asarray(ks)[np.argmax(bics <= umbral)])


def preparar_variables(df, columnas=("FREQ_MHz", "PW_us", "AOA_deg")):
    """Matriz de variables para clustering: log10 del PW y estandarización (z-score)."""
    from sklearn.preprocessing import StandardScaler
    X = df[list(columnas)].copy()
    if "PW_us" in X:
        X["PW_us"] = np.log10(X["PW_us"])
    return StandardScaler().fit_transform(X.values)


def ajustar_gmm(df, columnas=("FREQ_MHz", "PW_us", "AOA_deg"), k_max=12, n_muestra=8000, semilla=0, graficar=True):
    """Ajusta GMM eligiendo K por BIC (regla del codo). Devuelve (modelo, X, K)."""
    from sklearn.mixture import GaussianMixture
    X = preparar_variables(df, columnas)
    rng = np.random.default_rng(semilla)
    idx = rng.choice(len(X), min(n_muestra, len(X)), replace=False)
    ks = list(range(1, k_max + 1))
    bics = [GaussianMixture(k, covariance_type="full", n_init=2, random_state=semilla).fit(X[idx]).bic(X[idx])
            for k in ks]
    K = elegir_k_por_bic(bics, ks)
    if graficar:
        plt.figure(figsize=(8, 3.2))
        plt.plot(ks, bics, "o-"); plt.axvline(K, color="#C44E52", ls="--", label=f"K elegido = {K}")
        plt.xlabel("Nº de componentes K"); plt.ylabel("BIC (menor es mejor)"); plt.legend()
        plt.title("Selección del número de grupos por BIC"); plt.tight_layout(); plt.show()
    gmm = GaussianMixture(K, covariance_type="full", n_init=3, random_state=semilla).fit(X)
    return gmm, X, K


def nivel_confianza(candidatos):
    """Traduce el puntaje del mejor candidato a un nivel de confianza de identificación."""
    p1 = round(float(candidatos.loc[0, "PUNTAJE"]), 2)
    p2 = round(float(candidatos.loc[1, "PUNTAJE"]), 2) if len(candidatos) > 1 else 0.0
    if p1 >= 0.85 and round(p1 - p2, 2) >= 0.15:
        return "ALTA"
    if p1 >= 0.85:
        return "MEDIA (candidatos cercanos)"
    if p1 >= 0.65:
        return "BAJA"
    return "SIN CORRELACIÓN → DESCONOCIDO"
# ------------------------------------------------------------------
# F. EVALUACIÓN DE AMENAZA E INFORME (Lab 6)
# ------------------------------------------------------------------
def indicadores_amenaza(ficha):
    """Indicadores técnicos (no doctrinarios) que sugieren la función del emisor."""
    ind = []
    if str(ficha["SCAN_TIPO"]).startswith("Fijo"):
        ind.append("iluminación continua (posible lock-on)")
    if str(ficha["SCAN_TIPO"]).startswith("Cónico"):
        ind.append("scan cónico (seguimiento angular)")
    if ficha["PRF_MEDIA_Hz"] > 3000:
        ind.append("PRF alta (> 3 kHz)")
    if ficha["BANDA_IEEE"] in ("X", "Ku", "K", "Ka"):
        ind.append(f"banda {ficha['BANDA_IEEE']} (alta precisión angular)")
    if ficha["PW_MEDIANA_us"] < 0.5:
        ind.append("PW corto (< 0.5 µs, alta resolución en distancia)")
    if "Dwell" in str(ficha["PRI_TIPO"]):
        ind.append("Dwell & Switch (pulso-Doppler)")
    if ficha["RF_TIPO"] == "Ágil":
        ind.append("agilidad en frecuencia (EP)")
    return ind


def evaluar_amenaza(ficha, amenaza_biblioteca=None):
    """Nivel de amenaza preliminar: indicadores técnicos + biblioteca (si hay identificación)."""
    ind = indicadores_amenaza(ficha)
    seguimiento = any(("lock-on" in i) or ("cónico" in i) for i in ind)
    if seguimiento:
        return "ALTA", ind
    if amenaza_biblioteca in ("Alta", "Media", "Baja"):
        return amenaza_biblioteca.upper(), ind
    return ("MEDIA (a confirmar)" if len(ind) >= 2 else "BAJA (a confirmar)"), ind


def asociar_modos(tabla, tol_rf_mhz=10, tol_aoa_deg=6):
    """Busca grupos con la misma RF y AOA: posible MISMO emisor en distintos modos."""
    filas = []
    nombres = list(tabla.index)
    for i in range(len(nombres)):
        for j in range(i + 1, len(nombres)):
            a, b = tabla.loc[nombres[i]], tabla.loc[nombres[j]]
            d_rf = abs(a["RF_MEDIANA_MHz"] - b["RF_MEDIANA_MHz"])
            d_aoa = abs((a["AOA_MEDIA_deg"] - b["AOA_MEDIA_deg"] + 180) % 360 - 180)
            if d_rf <= tol_rf_mhz and d_aoa <= tol_aoa_deg:
                primero, segundo = (a, b) if a["T_INICIO_s"] <= b["T_INICIO_s"] else (b, a)
                filas.append({"GRUPO_1": primero.name, "GRUPO_2": segundo.name, "ΔRF_MHz": round(d_rf, 1),
                              "ΔAOA_deg": round(d_aoa, 1), "FIN_1_s": round(primero["T_FIN_s"], 2),
                              "INICIO_2_s": round(segundo["T_INICIO_s"], 2),
                              "SECUENCIAL": bool(segundo["T_INICIO_s"] >= primero["T_FIN_s"] - 0.5)})
    return pd.DataFrame(filas)


def tabla_md(df):
    """Convierte un DataFrame en una tabla Markdown (para el borrador del informe)."""
    cols = list(df.columns)
    lineas = ["| " + " | ".join(str(c) for c in cols) + " |", "|" + "---|" * len(cols)]
    for _, fila in df.iterrows():
        lineas.append("| " + " | ".join(str(v) for v in fila.values) + " |")
    return "\n".join(lineas)
from sklearn.mixture import GaussianMixture
os.makedirs('resultados/figuras', exist_ok=True)
print('Caja de herramientas lista ✔')

---
## Fase 1 · Metadatos y preprocesamiento

In [ ]:
ruta = asegurar_archivo(ARCHIVO)
meta = pd.read_excel(ruta, sheet_name="METADATOS").set_index("CAMPO")["VALOR"]
meta

In [ ]:
df = cargar_pdw(ARCHIVO, hoja="PDW")
INICIO = pd.to_datetime(meta["INICIO_COLECCION_UTC"])

def hora_utc(t_s):
    """Convierte segundos desde el inicio de la captura en hora UTC (texto)."""
    return (INICIO + pd.to_timedelta(t_s, unit="s")).strftime("%H:%M:%S.%f")[:-4] + "Z"

print(f"Inicio de colección: {INICIO} UTC · primer pulso {hora_utc(0)} · último pulso {hora_utc(df['TOA_s'].iloc[-1])}")

📘 `pd.to_datetime` convierte texto en una fecha/hora y `pd.to_timedelta` convierte segundos en una **duración**. Sumándolas obtenemos la **hora absoluta (UTC)** de cada evento: indispensable en un informe.

---
## Fase 2 · Panorama general

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(17, 9))
ax[0, 0].plot(df["TOA_s"], df["AMP_dBm"], ".", ms=1, color="tab:red"); ax[0, 0].set_ylabel("AMP (dBm)"); ax[0, 0].set_xlabel("Tiempo (s)")
ax[0, 1].plot(df["TOA_s"], df["FREQ_MHz"], ".", ms=1); ax[0, 1].set_ylabel("RF (MHz)"); ax[0, 1].set_xlabel("Tiempo (s)")
ax[1, 0].scatter(df["AOA_deg"], df["FREQ_MHz"], s=1, alpha=0.3); ax[1, 0].set_xlabel("AOA (°)"); ax[1, 0].set_ylabel("RF (MHz)")
ax[1, 1].scatter(df["FREQ_MHz"], df["PW_us"], s=1, alpha=0.3); ax[1, 1].set_yscale("log")
ax[1, 1].set_xlabel("RF (MHz)"); ax[1, 1].set_ylabel("PW (µs, log)")
fig.suptitle(f"{meta['MISION']} · panorama de la colección", fontsize=13)
plt.tight_layout(); plt.savefig("resultados/figuras/01_panorama.png", dpi=110); plt.show()

### ✏️ Primeras impresiones (2 minutos, antes de seguir)
¿Cuántos emisores crees que hay? ¿En qué bandas? ¿Ves algún comportamiento llamativo en el tiempo (aparición, desaparición, iluminación continua)?

*Respuesta del grupo:* …

---
## Fase 3 · Separación automática (GMM)

In [ ]:
gmm, X, K = ajustar_gmm(df, ("FREQ_MHz", "PW_us", "AOA_deg"), k_max=12)
etq = gmm.predict(X)
orden = pd.Series(df["FREQ_MHz"].values).groupby(etq).median().sort_values().index
mapa = {k: f"C{i+1}" for i, k in enumerate(orden)}
df["CLUSTER"] = [mapa[e] for e in etq]
df["P_MAX"] = gmm.predict_proba(X).max(axis=1)
print(f"K elegido por BIC: {K} · pulsos ambiguos (P<0,95): {(df['P_MAX'] < 0.95).sum()}")
df["CLUSTER"].value_counts().sort_index()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 5.5))
for c, sub in df.groupby("CLUSTER"):
    ax[0].scatter(sub["AOA_deg"], sub["FREQ_MHz"], s=2, label=c)
    ax[1].scatter(sub["FREQ_MHz"], sub["PW_us"], s=2, label=c)
ax[0].set_xlabel("AOA (°)"); ax[0].set_ylabel("RF (MHz)"); ax[0].legend(markerscale=5, ncol=2)
ax[1].set_yscale("log"); ax[1].set_xlabel("RF (MHz)"); ax[1].set_ylabel("PW (µs, log)")
fig.suptitle("Grupos del GMM"); plt.tight_layout(); plt.savefig("resultados/figuras/02_grupos_gmm.png", dpi=110); plt.show()

---
## Fase 4 · Caracterización y depuración del ruido

In [ ]:
fichas = pd.DataFrame([ficha_emisor(sub, c, verbose=False) for c, sub in df.groupby("CLUSTER")]).set_index("EMISOR")
ruido = fichas["PRI_TIPO"].str.startswith("Sin estructura")
print("Grupos descartados como ruido:", list(fichas.index[ruido]))
emis = fichas[~ruido].copy()
emis[["N_PULSOS", "T_INICIO_s", "T_FIN_s", "RF_MEDIANA_MHz", "RF_TIPO", "PW_MEDIANA_us", "AOA_MEDIA_deg",
      "PRI_TIPO", "PRI_DETALLE", "PRF_MEDIA_Hz", "SCAN_TIPO", "SCAN_DETALLE"]]

💡 **Revisa cada grupo** antes de seguir. Si dudas de alguno, analízalo en detalle con las herramientas del Lab 3, por ejemplo:

```python
sub = df[df["CLUSTER"] == "C3"]
graficar_pri(sub["TOA_us"], "C3"); analizar_pri(sub["TOA_us"]); analizar_scan(sub)
```

In [ ]:
# ✏️ Espacio para el análisis detallado de los grupos que el equipo considere necesario

---
## Fase 5 · ¿Un emisor, varios modos?
🎯 Dos grupos con la **misma RF y el mismo AOA** probablemente son el **mismo radar** operando en **modos distintos** (el GMM los separa porque cambió el PW). Si además son **secuenciales en el tiempo**, hubo un **cambio de modo**: un evento clave para el informe.

In [ ]:
pares = asociar_modos(emis)
pares

In [ ]:
eventos = []
for _, p in pares.iterrows():
    a, b = emis.loc[p["GRUPO_1"]], emis.loc[p["GRUPO_2"]]
    if p["SECUENCIAL"]:
        eventos.append({
            "t_s": b["T_INICIO_s"], "HORA_UTC": hora_utc(b["T_INICIO_s"]), "GRUPOS": f"{a.name} → {b.name}",
            "EVENTO": (f"CAMBIO DE MODO: {a['SCAN_TIPO']}, PRI {a['PRI_MEDIA_us']:.0f} µs, PW {a['PW_MEDIANA_us']} µs  →  "
                       f"{b['SCAN_TIPO']}, PRI {b['PRI_MEDIA_us']:.0f} µs (PRF {b['PRF_MEDIA_Hz']:.0f} Hz), PW {b['PW_MEDIANA_us']} µs"),
        })
# Apariciones / desapariciones "reales": más allá de 1,5 periodos de scan desde el borde de la captura
segundos_modo = set(pares.loc[pares["SECUENCIAL"], "GRUPO_2"]) if len(pares) else set()
primeros_modo = set(pares.loc[pares["SECUENCIAL"], "GRUPO_1"]) if len(pares) else set()
t_max = df["TOA_s"].max()
for nombre, f in emis.iterrows():
    T = f["SCAN_PERIODO_s"]
    margen = max(2.0, 1.5 * T) if pd.notna(T) else 2.0
    if f["T_INICIO_s"] > margen and nombre not in segundos_modo:
        eventos.append({"t_s": f["T_INICIO_s"], "HORA_UTC": hora_utc(f["T_INICIO_s"]), "GRUPOS": nombre,
                        "EVENTO": f"Aparece {nombre} ({f['RF_MEDIANA_MHz']:.0f} MHz, AOA {f['AOA_MEDIA_deg']:.0f}°, scan {f['SCAN_TIPO']})"})
    if f["T_FIN_s"] < t_max - margen and nombre not in primeros_modo:
        eventos.append({"t_s": f["T_FIN_s"], "HORA_UTC": hora_utc(f["T_FIN_s"]), "GRUPOS": nombre,
                        "EVENTO": f"Deja de detectarse {nombre}"})
eventos = pd.DataFrame(eventos).sort_values("t_s").reset_index(drop=True)
eventos

💡 Un emisor de scan lento puede tardar varios segundos en iluminar por primera vez al receptor sin que eso signifique que "apareció". Por eso sólo se registra una aparición/desaparición si ocurre a más de **1,5 periodos de scan** del borde de la captura.

---
## Fase 6 · Identificación contra la biblioteca

In [ ]:
bib = pd.read_excel(asegurar_archivo("BIBLIOTECA_EMISORES_ficticia.xlsx"))
ident = []
for nombre, f in emis.iterrows():
    f = f.copy(); f["EMISOR"] = nombre
    cand = comparar_con_biblioteca(f, bib)
    conf = nivel_confianza(cand)
    desconocido = conf.startswith("SIN")
    amen, ind = evaluar_amenaza(f, None if desconocido else cand.loc[0, "AMENAZA"])
    ident.append({
        "GRUPO": nombre,
        "IDENTIFICACION": "DESCONOCIDO" if desconocido else f"{cand.loc[0, 'ID_LIB']} {cand.loc[0, 'DESIGNACION']}",
        "FUNCION_PROBABLE": "—" if desconocido else cand.loc[0, "FUNCION"],
        "CONFIANZA": conf, "PUNTAJE": cand.loc[0, "PUNTAJE"],
        "2º_CANDIDATO": f"{cand.loc[1, 'ID_LIB']} ({cand.loc[1, 'PUNTAJE']})",
        "AMENAZA": amen, "INDICADORES": "; ".join(ind),
    })
ident = pd.DataFrame(ident).set_index("GRUPO")
ident

### ✏️ Revisión crítica del grupo (obligatoria)
La identificación automática es una **ayuda**, no una conclusión. Para cada fila respondan:
1. ¿Los dos grupos asociados en la Fase 5 recibieron la **misma** identificación? ¿Es coherente con la descripción de la biblioteca?
2. ¿Algún emisor tiene confianza **MEDIA** o **BAJA**? ¿Qué parámetro falló y qué medición adicional lo resolvería?
3. ¿Hay **desconocidos**? Asígnenles designación provisoria (`DESC-01`, …) y redacten su ficha.

*Respuestas del grupo:* …

---
## Fase 7 · Cronología y amenaza

In [ ]:
color_amenaza = {"ALTA": "#C44E52", "MEDIA": "#DD8452", "BAJA": "#55A868"}
filas_plot = list(emis.sort_values("RF_MEDIANA_MHz").index)
fig, ax = plt.subplots(figsize=(16, 0.7 * len(filas_plot) + 1.8))
for i, g in enumerate(filas_plot):
    sub = df[df["CLUSTER"] == g]
    nivel = ident.loc[g, "AMENAZA"].split(" ")[0]
    ax.plot(sub["TOA_s"], np.full(len(sub), i), "|", ms=14, color=color_amenaza.get(nivel, "gray"), alpha=0.6)
etiquetas_y = [f"{g} · {ident.loc[g, 'IDENTIFICACION'][:22]} · {emis.loc[g, 'RF_MEDIANA_MHz']:.0f} MHz" for g in filas_plot]
ax.set_yticks(range(len(filas_plot))); ax.set_yticklabels(etiquetas_y)
for _, e in eventos[eventos["EVENTO"].str.startswith("CAMBIO")].iterrows():
    ax.axvline(e["t_s"], color="k", ls="--")
    ax.text(e["t_s"], len(filas_plot) - 0.55, f"cambio de modo {e['HORA_UTC']}  ", fontsize=9, ha="right")
ax.set_xlabel("Tiempo desde el inicio de la colección (s)")
ax.set_title("Cronología de emisores (color = nivel de amenaza preliminar: rojo alta · naranja media · verde baja)")
plt.tight_layout(); plt.savefig("resultados/figuras/03_cronologia.png", dpi=110); plt.show()

In [ ]:
# Mapa polar de marcaciones (AOA) con el nivel de amenaza
fig = plt.figure(figsize=(7, 7)); axp = fig.add_subplot(projection="polar")
axp.set_theta_zero_location("N"); axp.set_theta_direction(-1)
etiquetas = {}                                   # agrupa grupos con la misma marcación (p. ej. dos modos)
for g, f in emis.iterrows():
    nivel = ident.loc[g, "AMENAZA"].split(" ")[0]
    axp.plot(np.deg2rad(f["AOA_MEDIA_deg"]), 1, "o", ms=14, color=color_amenaza.get(nivel, "gray"))
    clave = round(f["AOA_MEDIA_deg"] / 3)
    etiquetas.setdefault(clave, [f["AOA_MEDIA_deg"], [], f["RF_MEDIANA_MHz"]])[1].append(g)
for aoa, grupos, rf in etiquetas.values():
    axp.annotate(f"{'/'.join(sorted(grupos))}\n{rf:.0f} MHz", (np.deg2rad(aoa), 1), xytext=(np.deg2rad(aoa), 1.27),
                 ha="center", fontsize=9)
axp.set_rticks([]); axp.set_ylim(0, 1.45); axp.set_title("Marcaciones de los emisores (AOA, 0° = proa/norte)")
plt.savefig("resultados/figuras/04_marcaciones.png", dpi=110, bbox_inches="tight"); plt.show()

⚠️ El AOA es la marcación **relativa al receptor**. Para ubicar un emisor en el mapa se necesitan varias marcaciones desde posiciones distintas (triangulación) o el rumbo/posición de la plataforma.

---
## Fase 8 · Anexo técnico y borrador automático

In [ ]:
emisores_tabla = emis.join(ident)[[
    "IDENTIFICACION", "FUNCION_PROBABLE", "CONFIANZA", "AMENAZA", "N_PULSOS", "T_INICIO_s", "T_FIN_s",
    "RF_MEDIANA_MHz", "RF_MIN_MHz", "RF_MAX_MHz", "RF_TIPO", "BANDA_IEEE", "BANDA_OTAN", "PW_MEDIANA_us", "PW_P5_us", "PW_P95_us",
    "PRI_TIPO", "PRI_DETALLE", "PRI_MEDIA_us", "PRF_MEDIA_Hz", "SCAN_TIPO", "SCAN_PERIODO_s", "SCAN_DETALLE",
    "AOA_MEDIA_deg", "AMP_MAX_dBm", "INDICADORES"]].copy()
emisores_tabla.insert(5, "PRIMERA_DET_UTC", [hora_utc(t) for t in emisores_tabla["T_INICIO_s"]])
emisores_tabla.insert(6, "ULTIMA_DET_UTC", [hora_utc(t) for t in emisores_tabla["T_FIN_s"]])

anexo = f"resultados/IPI_GRUPO_{GRUPO}_anexo_tecnico.xlsx"
with pd.ExcelWriter(anexo) as w:
    meta.to_frame().to_excel(w, sheet_name="METADATOS")
    emisores_tabla.to_excel(w, sheet_name="EMISORES")
    eventos.to_excel(w, sheet_name="EVENTOS", index=False)
    pares.to_excel(w, sheet_name="ASOCIACION_MODOS", index=False)
    fichas.to_excel(w, sheet_name="TODOS_LOS_GRUPOS")
print("Anexo técnico:", anexo)

In [ ]:
resumen = emisores_tabla.reset_index()[["EMISOR", "IDENTIFICACION", "CONFIANZA", "AMENAZA", "RF_MEDIANA_MHz", "PRI_TIPO",
                                        "PRF_MEDIA_Hz", "PW_MEDIANA_us", "SCAN_TIPO", "AOA_MEDIA_deg", "PRIMERA_DET_UTC", "ULTIMA_DET_UTC"]]
borrador = f"""# INFORME PRELIMINAR DE INTERCEPTACIÓN — BORRADOR AUTOMÁTICO (GRUPO {GRUPO})
**{meta['CLASIFICACION']}**

## 1. Datos generales
- Misión: {meta['MISION']}
- Plataforma colectora: {meta['PLATAFORMA_COLECTORA']} · Sensor: {meta['SENSOR']}
- Área: {meta['AREA']} · Inicio de colección: {meta['INICIO_COLECCION_UTC']} UTC · Duración: {meta['DURACION_s']} s
- Pulsos válidos analizados: {len(df):,} · Grupos del GMM: {K} · Descartados como ruido: {int(ruido.sum())}

## 3. Emisores interceptados
{tabla_md(resumen)}

## 4. Eventos (cronológico)
{tabla_md(eventos[["HORA_UTC", "GRUPOS", "EVENTO"]])}

## 2 / 5 / 6 / 7. [COMPLETAR POR EL ANALISTA: resumen ejecutivo, desconocidos, evaluación, recomendaciones]
"""
with open(f"resultados/IPI_GRUPO_{GRUPO}_borrador.md", "w", encoding="utf-8") as fh:
    fh.write(borrador)
print(borrador)

💡 En Colab, descarga los productos con:
```python
from google.colab import files
files.download(f"resultados/IPI_GRUPO_{GRUPO}_anexo_tecnico.xlsx")
```

---
## Fase 9 · Redacción del informe

### 📘 Lenguaje de estimación
Un informe de inteligencia **separa lo observado de lo inferido** y gradúa la certeza:

| Nivel de confianza | Expresión sugerida | Cuándo |
|---|---|---|
| ALTA | "Se identifica…", "corresponde a…" | Todos los parámetros calzan con una sola entrada; scan observado completo |
| MEDIA | "Probablemente…", "es consistente con…" | Calzan los parámetros principales; candidatos cercanos o scan incompleto |
| BAJA | "Posiblemente…", "no se descarta…" | Calce parcial, pocos pulsos o captura corta |
| Sin correlación | "Emisor no catalogado (DESC-xx)…" | Ninguna entrada calza |

### ✅ Checklist de calidad antes de entregar
- [ ] El **BLUF** (resumen ejecutivo) responde en 3–5 líneas: ¿qué hay, dónde, qué amenaza, qué cambió?
- [ ] Todos los valores llevan **unidad** y, cuando corresponde, **incertidumbre** (IC 95 % o P5–P95).
- [ ] Las horas están en **UTC** y son coherentes con los metadatos.
- [ ] Cada identificación tiene su **nivel de confianza** y su justificación.
- [ ] Los **eventos** (cambios de modo, apariciones) están en orden cronológico.
- [ ] Los **desconocidos** tienen designación provisoria y ficha completa.
- [ ] Se declaran las **limitaciones** (duración de captura, pulsos ambiguos, scans incompletos, precisión del sensor).
- [ ] Las **recomendaciones** son concretas (qué colectar, dónde, cuándo, a quién difundir).

---
<div style="border:2px solid #1F4E79;padding:14px">

# INFORME PRELIMINAR DE INTERCEPTACIÓN (IPI) Nº ___ / 2026

**Clasificación:** MATERIAL DE INSTRUCCIÓN – DATOS SINTÉTICOS
**DTG del informe:** ________Z · **Grupo / analistas:** ______________________

### 1. DATOS GENERALES
| Campo | Valor |
|---|---|
| Misión | |
| Plataforma colectora / sensor | |
| Área | |
| Periodo de colección (UTC) | |
| Pulsos analizados / descartados | |

### 2. RESUMEN EJECUTIVO (BLUF)
- …
- …
- …

### 3. EMISORES INTERCEPTADOS
| ID | Identificación (confianza) | Función probable | RF (MHz) | PRI / PRF | PW (µs) | Scan | AOA (°) | 1ª / última detección (UTC) | Amenaza |
|---|---|---|---|---|---|---|---|---|---|
| | | | | | | | | | |

### 4. EVENTOS RELEVANTES (orden cronológico)
| Hora UTC | Emisor | Evento | Implicancia |
|---|---|---|---|
| | | | |

### 5. EMISORES NO CATALOGADOS
**DESC-01:** parámetros completos (RF, PRI y tipo, PW, scan, AOA, nº de pulsos, horas) · semejanzas con entradas de la biblioteca · hipótesis de función.

### 6. EVALUACIÓN DEL ANALISTA
- Interpretación del escenario (qué sistemas, qué disposición, qué nivel de actividad / alerta):
- Limitaciones del análisis:

### 7. RECOMENDACIONES
- Colección adicional:
- Actualización de biblioteca:
- Difusión:

### 8. ANEXOS
- A. Anexo técnico (`resultados/IPI_GRUPO_n_anexo_tecnico.xlsx`)
- B. Figuras (`resultados/figuras/`)
</div>

---
## Pauta de evaluación del IPI

| Criterio | Peso | Logrado (100 %) | Parcial (50 %) | No logrado (0 %) |
|---|---|---|---|---|
| Preprocesamiento y trazabilidad | 10 % | Bitácora completa, decisiones justificadas | Limpieza correcta sin justificar | Datos sin depurar o pasos no documentados |
| Separación de emisores y ruido | 20 % | Todos los emisores separados, ruido identificado, modos asociados | Errores menores (un grupo fusionado/partido) | Separación incorrecta |
| Caracterización técnica | 25 % | RF, PRI (tipo y valores), PW, scan y AOA con unidades e incertidumbre | Parámetros correctos sin incertidumbre o con un error | Varios parámetros erróneos |
| Identificación y confianza | 15 % | Identificación correcta con nivel de confianza justificado; desconocido bien tratado | Identificación correcta sin justificación | Identificaciones erróneas o sin confianza |
| Eventos y amenaza | 15 % | Cambio de modo detectado con hora UTC; evaluación de amenaza fundamentada | Evento detectado sin hora o sin implicancia | Evento no detectado |
| Calidad del informe | 15 % | BLUF claro, lenguaje de estimación, recomendaciones concretas | Informe completo pero confuso | Incompleto |

### Preguntas finales de reflexión
1. ¿Qué parte del proceso fue más dependiente del **criterio del analista** y cuál podría automatizarse por completo?
2. Si la colección hubiera durado sólo 10 s, ¿qué conclusiones de tu informe cambiarían o perderían confianza?
3. ¿Qué información adicional (de otro sensor u otra fuente) te habría permitido elevar el nivel de confianza?

---
### ✅ Fin del curso de laboratorios

Recorriste el ciclo completo de producción de inteligencia de señales de radar: **datos crudos → preprocesamiento → estadística → caracterización → separación → identificación → informe**. Las mismas herramientas funcionan con capturas reales: sólo cambia el archivo de entrada (y quizás los nombres de columnas).

<span style="font-size:11px;color:#7a7a7a">Curso de Guerra Electrónica · Análisis de Señales de Radar · Material de instrucción con datos sintéticos y biblioteca ficticia.</span>